# Susceptible-Infected (SI) on an Erdos-Renyi Graph

This notebook mirrors `03_IC_on_complete_graph.ipynb`, but uses the **SI** diffusion model on a **random Erdos-Renyi (ER)** contact network.

---

## What is the SI model?

1. Start with one **source** (Patient Zero) infected at `t=0`.
2. At **each timestep**, every **infected** node tries to infect each **susceptible** neighbour with probability **beta**.
3. Infected nodes **never recover** (no SIR-style recovery).
4. The process stops when **no new infections occur in a timestep**.

**Implementation note (matches `src.data.cascade.SIModel`):** updates are **discrete-time and simultaneous**. If, at some timestep, **every** Bernoulli transmission fails, the simulation ends there even if susceptible nodes remain. So runs can **stall** before covering the whole graph, especially when `beta` is small. Report both **final cascade size** and **last infection timestep**.

---

## Why Erdos-Renyi?

ER graphs `G(n, p)` are a standard **unstructured baseline**: edges appear at random, degrees are concentrated around `p(n-1)`. Compared to a complete graph, nodes are **not** symmetric, so cascade **shape** and **timing** carry more geometric information.


## Parameters — edit these to experiment

| Parameter | Meaning | Try changing it to... |
|-----------|---------|------------------------|
| `N` | Number of nodes | 20, 40, 100 |
| `P_ER` | ER edge probability `p` in `G(n, p)` | 0.05 (sparse), 0.15 (denser) |
| `GRAPH_SEED` | RNG seed for graph generation | any integer |
| `SOURCE` | Patient Zero | any node id `0 .. N-1` (must exist) |
| `SEED` | RNG seed for one SI run | change for different stochastic outcomes |
| `R0` | Map to `beta` via `beta = R0 / <k>` (if `USE_R0`) | 0.5, 1.0, 2.0, ... |
| `MAX_STEPS` | Safety cap on SI timesteps | raise if runs stall before full spread |


In [ ]:
# --- Parameters — change these and re-run! ---

N          = 40          # Nodes in G(n, p)
P_ER       = 0.12       # Edge probability (higher => denser)
GRAPH_SEED = 42         # Reproducible ER instance

SOURCE = 3              # Patient Zero
SEED   = 43             # Seed for the SI simulation below

USE_R0 = True           # True: set beta from R0 and <k>; False: set beta directly
R0     = 2.0            # Target basic reproduction number (used if USE_R0)
beta   = 0.15           # Transmission probability per edge per timestep (if not USE_R0)

MAX_STEPS = 2000        # SI timestep limit (raise if infection is slow)


## Step 1 — Build a connected Erdos-Renyi graph


In [ ]:
import sys
sys.path.insert(0, '..')

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from src.data.networks import generate_er_network, compute_network_stats
from src.data.cascade import r0_to_params

G = generate_er_network(n=N, p=P_ER, seed=GRAPH_SEED)
stats = compute_network_stats(G)
avg_degree = float(stats["avg_degree"])

if USE_R0:
    params = r0_to_params(R0, avg_degree, model="SI")
    beta = params["beta"]
    print(f"R0={R0}  ->  beta = R0 / <k> = {R0} / {avg_degree:.2f} = {beta:.4f}")
else:
    print(f"Using beta={beta} directly")

print("\nNetwork statistics:")
for k, v in stats.items():
    print(f"  {k:<20} {v}")


In [ ]:
# Layout: spring embedding (ER is not regular — avoid circular layout)
pos = nx.spring_layout(G, seed=GRAPH_SEED, k=0.85)

fig, ax = plt.subplots(figsize=(7, 7))
node_colors = ['tomato' if n == SOURCE else 'steelblue' for n in G.nodes()]
node_sizes  = [520 if n == SOURCE else 260 for n in G.nodes()]

nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, width=0.7)
nx.draw_networkx_nodes(
    G, pos, ax=ax, node_color=node_colors, node_size=node_sizes,
    edgecolors='black', linewidths=1.2,
)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_color='white')

ax.set_title(
    f'Erdos-Renyi G({N}, {P_ER})  —  source {SOURCE} (red)',
    fontsize=13, fontweight='bold',
)
ax.axis('off')
plt.tight_layout()
plt.show()


## Step 2 — Run the SI simulation

`SIModel` runs discrete-time SI until a timestep with no new infections.

Outputs (`CascadeResult`): `source`, `infection_times`, `cascade_edges`, `observed_graph` (undirected), `size`, `depth`.


In [ ]:
from src.data.cascade import SIModel

model  = SIModel(beta=beta)
result = model.run(G, source=SOURCE, seed=SEED, max_steps=MAX_STEPS)

last_t = max(result.infection_times.values(), default=0)
print("=== SI Cascade Result ===")
print(f"  Source (Patient Zero) : {result.source}")
print(f"  Infected nodes        : {result.size} / {G.number_of_nodes()}")
print(f"  Last infection time   : t = {last_t}")
print(f"  Cascade depth (hops)  : {result.depth}")
print(f"  Empirical R0 (tree)   : {result.actual_r0():.2f}")

by_time = {}
for node, t in sorted(result.infection_times.items(), key=lambda x: x[1]):
    by_time.setdefault(t, []).append(node)
print("\n  Infection timeline (first few timesteps):")
for t, nodes in sorted(by_time.items())[:8]:
    tag = '  <- SOURCE' if t == 0 else ''
    print(f"    t={t}: {len(nodes)} nodes{tag}")
if len(by_time) > 8:
    print(f"    ... ({len(by_time)} timesteps total)")


## Step 3 — Visualise the cascade tree

Directed edges show who infected whom (ground truth — not assumed visible to ML).


In [ ]:
from src.visualization.cascades import plot_cascade_tree

fig = plot_cascade_tree(result, G, figsize=(9, 6))
plt.show()


## Step 4 — Visualise on the ER substrate

Node colour encodes infection time; red = source. Right panel overlays transmission edges from the infection tree.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

times = result.infection_times
max_t = max(times.values(), default=1) or 1
cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=0, vmax=max_t)
infected = set(times.keys())
uninfected = set(G.nodes()) - infected


def draw_network(ax, show_cascade_edges=False):
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.12, width=0.5, edge_color='gray')
    if show_cascade_edges and result.cascade_edges:
        nx.draw_networkx_edges(
            result.infection_tree, pos, ax=ax,
            edge_color='crimson', width=2.0, alpha=0.85,
            arrows=True, arrowsize=16,
            connectionstyle='arc3,rad=0.08',
        )
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        nodelist=[n for n in infected if n != SOURCE],
        node_color=[cmap(norm(times[n])) for n in infected if n != SOURCE],
        node_size=320, edgecolors='black', linewidths=0.6,
    )
    nx.draw_networkx_nodes(
        G, pos, ax=ax, nodelist=[SOURCE],
        node_color='red', node_size=520,
        edgecolors='black', linewidths=2,
    )
    nx.draw_networkx_nodes(
        G, pos, ax=ax, nodelist=list(uninfected),
        node_color='lightgray', node_size=220,
        edgecolors='black', linewidths=0.5,
    )
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color='black')


draw_network(axes[0], show_cascade_edges=False)
axes[0].set_title(
    'Infection time on ER\n(red=source, yellow-red=infected, gray=susceptible)',
    fontsize=11, fontweight='bold',
)
axes[0].axis('off')

draw_network(axes[1], show_cascade_edges=True)
axes[1].set_title('With transmission edges', fontsize=11, fontweight='bold')
axes[1].axis('off')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.03, pad=0.02).set_label(
    'Infection timestep', fontsize=11
)

fig.suptitle(
    f'SI on ER  |  beta={beta:.4f}  |  infected={result.size}/{G.number_of_nodes()}  '
    f'|  last_t={last_t}  |  depth={result.depth}',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.show()


## Step 5 — Many runs: cascade size and duration

Track **final infected count** (`size`) and the **last infection timestep** `max(infection_times.values())`. Full-graph runs show how fast the cascade moved; early stops show how often the discrete-time SI stalls.


In [ ]:
import numpy as np

N_RUNS = 200

durations = []
sizes = []
for seed_i in range(N_RUNS):
    r = SIModel(beta=beta).run(G, source=SOURCE, seed=seed_i, max_steps=MAX_STEPS)
    sizes.append(r.size)
    durations.append(max(r.infection_times.values(), default=0))

n_nodes = G.number_of_nodes()
full = sum(1 for s in sizes if s == n_nodes)
print(f"Over {N_RUNS} runs (beta={beta:.4f}, same G and SOURCE):")
print(f"  Runs infecting all {n_nodes} nodes: {full} ({100*full/N_RUNS:.0f}%)")
print(f"  Mean final size     : {np.mean(sizes):.1f} / {n_nodes}")
print(f"  Mean last timestep  : {np.mean(durations):.1f}")
print(f"  Median last timestep: {np.median(durations):.1f}")
print(f"  Min / max last t    : {min(durations)} / {max(durations)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sizes, bins=range(1, n_nodes + 2), align='left', color='coral',
             edgecolor='white', linewidth=0.5, rwidth=0.85)
axes[0].set_xlabel('Final cascade size (# infected)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Final size distribution', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(1, n_nodes + 1, max(1, n_nodes // 10)))

axes[1].hist(durations, bins='auto', color='steelblue', edgecolor='white', linewidth=0.5)
axes[1].axvline(np.mean(durations), color='tomato', linewidth=2, linestyle='--',
                label=f'Mean = {np.mean(durations):.1f}')
axes[1].set_xlabel('Last infection timestep', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Last infection time', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
fig.suptitle(f'SI on ER — G({N},{P_ER}), beta={beta:.4f}, {N_RUNS} runs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 6 — Sweep R0 and mean duration

We remap each `R0` to `beta = R0 / <k>` using the **same** ER graph's `<k>`.


In [ ]:
R0_SWEEP  = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
N_RUNS_SW = 120

mean_durations = []
for r0 in R0_SWEEP:
    b = r0_to_params(r0, avg_degree, model="SI")["beta"]
    durs, sz = [], []
    for i in range(N_RUNS_SW):
        r = SIModel(beta=b).run(G, source=SOURCE, seed=1000 + i, max_steps=MAX_STEPS)
        sz.append(r.size)
        durs.append(max(r.infection_times.values(), default=0))
    mean_durations.append(np.mean(durs))
    print(
        f"R0={r0:.1f}  beta={b:.4f}  mean_size={np.mean(sz):.1f}/{G.number_of_nodes()}  "
        f"mean_last_t={np.mean(durs):.1f}"
    )

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(R0_SWEEP, mean_durations, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('R0', fontsize=12)
ax.set_ylabel('Mean last infection timestep', fontsize=12)
ax.set_title(
    f'R0 vs mean SI duration — ER graph (N={N}, p={P_ER}), {N_RUNS_SW} runs/R0',
    fontsize=13, fontweight='bold',
)
ax.axvline(1.0, color='tomato', linestyle='--', alpha=0.7, label='R0 = 1')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
## Things to try

1. **Change `P_ER`** — sparser graphs (smaller `p`) have longer paths; SI may need more timesteps.
2. **Change `SOURCE`** — pick a low- vs high-degree node (`G.degree(SOURCE)`) and compare duration.
3. **Compare to `03_IC_on_complete_graph.ipynb`** — IC can die out early; here SI can **stall** if a whole timestep has no successful transmissions.
4. **Raise `MAX_STEPS`** if prints show sizes below `N` (infection still ongoing when capped).
